In [9]:
import requests
import json
import time
from datetime import datetime
from urllib.parse import quote

# 세션 생성 (쿠키 유지용)
session = requests.Session()

def method_1_direct_page_access():
    """방법 1: 직접 입찰공고 목록 페이지에 접근"""
    
    print("=== 방법 1: 직접 페이지 접근 ===")
    
    # 실제 입찰 목록 페이지에 직접 접근
    list_page_url = 'https://www.g2b.go.kr/ep/tbid/tbidList.do'
    
    headers = {
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8',
        'Accept-Language': 'ko-KR,ko;q=0.9,en-US;q=0.8,en;q=0.7',
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/131.0.0.0 Safari/537.36',
        'sec-ch-ua': '"Google Chrome";v="131", "Chromium";v="131", "Not_A Brand";v="24"',
        'sec-ch-ua-mobile': '?0',
        'sec-ch-ua-platform': '"Windows"',
        'sec-fetch-dest': 'document',
        'sec-fetch-mode': 'navigate',
        'sec-fetch-site': 'none',
        'sec-fetch-user': '?1'
    }
    
    try:
        response = session.get(list_page_url, headers=headers, timeout=30)
        print(f"페이지 상태 코드: {response.status_code}")
        print(f"쿠키: {session.cookies.get_dict()}")
        
        if response.status_code == 200:
            # HTML에서 필요한 토큰이나 정보 추출
            html_content = response.text
            
            # CSRF 토큰 찾기
            import re
            csrf_pattern = r'name="?_token"?\s+value="?([^"\s]+)"?'
            csrf_match = re.search(csrf_pattern, html_content)
            
            if csrf_match:
                csrf_token = csrf_match.group(1)
                print(f"CSRF 토큰 발견: {csrf_token}")
                return csrf_token
            else:
                print("CSRF 토큰을 찾을 수 없습니다.")
                
            # 페이지 내용에서 필요한 정보 찾기
            if 'JSESSIONID' in str(session.cookies):
                print("세션 ID 획득 성공")
            
            return True
        else:
            print(f"페이지 접근 실패: {response.status_code}")
            return False
            
    except requests.exceptions.RequestException as e:
        print(f"페이지 접근 오류: {e}")
        return False

def method_2_different_endpoint():
    """방법 2: 다른 엔드포인트 시도"""
    
    print("\n=== 방법 2: 다른 엔드포인트 시도 ===")
    
    # 일반 검색 엔드포인트 시도
    search_url = 'https://www.g2b.go.kr/pn/pnp/pnpe/BidPbac/selectBidPbacList.do'
    
    headers = {
        'Accept': 'application/json, text/javascript, */*; q=0.01',
        'Accept-Language': 'ko-KR,ko;q=0.9,en-US;q=0.8,en;q=0.7',
        'Content-Type': 'application/x-www-form-urlencoded; charset=UTF-8',
        'Origin': 'https://www.g2b.go.kr',
        'Referer': 'https://www.g2b.go.kr/ep/tbid/tbidList.do',
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/131.0.0.0 Safari/537.36',
        'X-Requested-With': 'XMLHttpRequest'
    }
    
    # Form 데이터 형식으로 전송
    form_data = {
        'dlBidPbancLstM.bidDateType': 'R',
        'dlBidPbancLstM.fromBidDt': '20241201',
        'dlBidPbancLstM.toBidDt': '20241231',
        'dlBidPbancLstM.bidPbancNm': '',
        'dlBidPbancLstM.startIndex': '1',
        'dlBidPbancLstM.endIndex': '10',
        'dlBidPbancLstM.recordCountPerPage': '10',
        'dlBidPbancLstM.currentPage': '1'
    }
    
    try:
        response = session.post(search_url, data=form_data, headers=headers, timeout=30)
        print(f"검색 상태 코드: {response.status_code}")
        
        if response.status_code == 200:
            try:
                json_data = response.json()
                with open('g2b_search_result.json', 'w', encoding='utf-8') as f:
                    json.dump(json_data, f, ensure_ascii=False, indent=2)
                print("일반 검색 성공! 'g2b_search_result.json' 저장됨")
                return True
            except json.JSONDecodeError:
                print("JSON이 아닌 응답:", response.text[:200])
                return False
        else:
            print(f"검색 실패: {response.text[:200]}")
            return False
            
    except requests.exceptions.RequestException as e:
        print(f"검색 오류: {e}")
        return False

def method_3_selenium_approach():
    """방법 3: Selenium 사용 제안"""
    
    print("\n=== 방법 3: Selenium 접근 제안 ===")
    
    selenium_code = '''
# pip install selenium 필요
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import json
import time

def get_g2b_with_selenium():
    # Chrome 드라이버 설정
    options = webdriver.ChromeOptions()
    options.add_argument('--headless')  # 백그라운드 실행
    options.add_argument('--no-sandbox')
    options.add_argument('--disable-dev-shm-usage')
    
    driver = webdriver.Chrome(options=options)
    
    try:
        # 나라장터 입찰공고 페이지 접속
        driver.get('https://www.g2b.go.kr/ep/tbid/tbidList.do')
        
        # 페이지 로딩 대기
        WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.ID, "searchForm"))
        )
        
        # 검색 조건 설정
        # 여기서 필요한 검색 조건들을 설정하고 검색 버튼 클릭
        
        # 결과 데이터 추출
        # ...
        
        return True
        
    except Exception as e:
        print(f"Selenium 오류: {e}")
        return False
        
    finally:
        driver.quit()
'''
    
    print("Selenium을 사용한 접근 방법:")
    print(selenium_code)
    
    return False

def method_4_rss_feed():
    """방법 4: RSS 피드나 공개 API 확인"""
    
    print("\n=== 방법 4: RSS 피드 확인 ===")
    
    # 나라장터 RSS 피드 시도
    rss_urls = [
        'https://www.g2b.go.kr/rss/getBidRss.do',
        'https://www.g2b.go.kr/rss/getBidRss.do?category=bidding',
        'https://www.g2b.go.kr/api/getBidList.do'  # 추측
    ]
    
    for rss_url in rss_urls:
        try:
            print(f"RSS 피드 시도: {rss_url}")
            response = session.get(rss_url, timeout=10)
            print(f"상태 코드: {response.status_code}")
            
            if response.status_code == 200:
                print(f"RSS 응답 (처음 200자): {response.text[:200]}")
                if response.text.strip():
                    with open(f'g2b_rss_{rss_url.split("/")[-1]}.xml', 'w', encoding='utf-8') as f:
                        f.write(response.text)
                    print(f"RSS 데이터 저장됨")
                    return True
            else:
                print(f"RSS 실패: {response.status_code}")
                
        except requests.exceptions.RequestException as e:
            print(f"RSS 요청 오류: {e}")
            
    return False

# 모든 방법 시도
def try_all_methods():
    print("나라장터 데이터 수집 - 대안 방법들 시도\n")
    
    # 방법 1: 직접 페이지 접근
    if method_1_direct_page_access():
        time.sleep(2)
        
        # 페이지 접근 후 다시 API 시도
        print("\n페이지 접근 후 API 재시도...")
        # 여기에 원래 API 호출 코드 실행
    
    # 방법 2: 다른 엔드포인트
    if method_2_different_endpoint():
        print("방법 2 성공!")
        return True
        
    # 방법 3: Selenium 제안
    method_3_selenium_approach()
    
    # 방법 4: RSS 피드
    if method_4_rss_feed():
        print("방법 4 성공!")
        return True
        
    print("\n모든 방법이 실패했습니다.")
    print("나라장터에서 추가적인 보안 검증을 하고 있는 것 같습니다.")
    print("\n추천 해결 방법:")
    print("1. Selenium을 사용한 브라우저 자동화")
    print("2. 나라장터 공식 API나 데이터 다운로드 서비스 이용")
    print("3. 브라우저에서 수동으로 데이터 다운로드 후 파일 처리")
    
    return False

# 실행
if __name__ == "__main__":
    try_all_methods()

나라장터 데이터 수집 - 대안 방법들 시도

=== 방법 1: 직접 페이지 접근 ===
페이지 상태 코드: 200
쿠키: {}
CSRF 토큰을 찾을 수 없습니다.

페이지 접근 후 API 재시도...

=== 방법 2: 다른 엔드포인트 시도 ===
검색 상태 코드: 500
검색 실패: {"ErrorMsg":"조회하는데 에러가 발생하였습니다.","reason":"Internal Server Error","path":"/pn/pnp/pnpe/BidPbac/selectBidPbacList.do","ErrorCode":-1,"locale":"ko"}

=== 방법 3: Selenium 접근 제안 ===
Selenium을 사용한 접근 방법:

# pip install selenium 필요
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import json
import time

def get_g2b_with_selenium():
    # Chrome 드라이버 설정
    options = webdriver.ChromeOptions()
    options.add_argument('--headless')  # 백그라운드 실행
    options.add_argument('--no-sandbox')
    options.add_argument('--disable-dev-shm-usage')

    driver = webdriver.Chrome(options=options)

    try:
        # 나라장터 입찰공고 페이지 접속
        driver.get('https://www.g2b.go.kr/ep/tbid/tbidList.do')

        # 

In [10]:
# RSS 파일 내용 확인 및 처리
import json
import xml.etree.ElementTree as ET
import requests
from datetime import datetime
import re

def check_rss_file():
    """저장된 RSS 파일 내용 확인"""
    try:
        with open('g2b_rss_getBidRss.do.xml', 'r', encoding='utf-8') as f:
            content = f.read()
            
        print("=== RSS 파일 내용 분석 ===")
        print(f"파일 크기: {len(content)} bytes")
        print(f"처음 500자:\n{content[:500]}")
        
        # HTML인지 XML인지 확인
        if content.strip().startswith('<!DOCTYPE html>') or content.strip().startswith('<html'):
            print("\n이것은 HTML 페이지입니다. RSS XML이 아닙니다.")
            return False
        elif content.strip().startswith('<?xml') or '<rss' in content:
            print("\n이것은 XML/RSS 데이터입니다.")
            return parse_rss_xml(content)
        else:
            print("\n알 수 없는 형식입니다.")
            return False
            
    except FileNotFoundError:
        print("RSS 파일을 찾을 수 없습니다.")
        return False
    except Exception as e:
        print(f"파일 읽기 오류: {e}")
        return False

def parse_rss_xml(xml_content):
    """RSS XML 파싱"""
    try:
        root = ET.fromstring(xml_content)
        
        # RSS 채널 정보
        channel = root.find('channel')
        if channel is not None:
            title = channel.find('title')
            description = channel.find('description')
            
            print(f"RSS 제목: {title.text if title is not None else 'N/A'}")
            print(f"RSS 설명: {description.text if description is not None else 'N/A'}")
            
            # RSS 아이템들
            items = channel.findall('item')
            print(f"RSS 아이템 수: {len(items)}")
            
            parsed_data = []
            for i, item in enumerate(items[:10]):  # 처음 10개만
                item_data = {}
                for child in item:
                    item_data[child.tag] = child.text
                parsed_data.append(item_data)
                
                # 첫 번째 아이템 미리보기
                if i == 0:
                    print(f"\n첫 번째 입찰 정보:")
                    for key, value in item_data.items():
                        print(f"- {key}: {value}")
            
            # JSON으로 저장
            with open('g2b_rss_parsed.json', 'w', encoding='utf-8') as f:
                json.dump(parsed_data, f, ensure_ascii=False, indent=2)
            print(f"\n파싱된 데이터를 'g2b_rss_parsed.json'에 저장했습니다.")
            
            return True
        else:
            print("RSS channel을 찾을 수 없습니다.")
            return False
            
    except ET.ParseError as e:
        print(f"XML 파싱 오류: {e}")
        return False
    except Exception as e:
        print(f"RSS 파싱 오류: {e}")
        return False

def try_proper_rss_feeds():
    """실제 RSS 피드 URL들을 시도"""
    
    print("\n=== 실제 RSS 피드 시도 ===")
    
    # 나라장터의 실제 RSS 피드 URL들 (추측)
    rss_feeds = [
        'https://www.g2b.go.kr/rss/rss.do',
        'https://www.g2b.go.kr/rss/rss.do?category=bid',
        'https://www.g2b.go.kr/rss/getBidRss.do?rss=1',
        'https://www.g2b.go.kr/api/rss/bidList.xml',
        'https://www.g2b.go.kr/openapi/rss/bidList.do'
    ]
    
    session = requests.Session()
    
    for i, rss_url in enumerate(rss_feeds, 1):
        try:
            print(f"\n{i}. {rss_url} 시도 중...")
            
            headers = {
                'Accept': 'application/rss+xml, application/xml, text/xml',
                'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
            }
            
            response = session.get(rss_url, headers=headers, timeout=10)
            print(f"   상태 코드: {response.status_code}")
            
            if response.status_code == 200:
                content = response.text.strip()
                print(f"   응답 길이: {len(content)} bytes")
                print(f"   응답 시작: {content[:100]}...")
                
                # XML/RSS인지 확인
                if content.startswith('<?xml') or '<rss' in content[:100]:
                    print(f"   ✓ RSS/XML 데이터 발견!")
                    
                    filename = f'g2b_rss_feed_{i}.xml'
                    with open(filename, 'w', encoding='utf-8') as f:
                        f.write(content)
                    print(f"   저장됨: {filename}")
                    
                    # 파싱 시도
                    if parse_rss_xml(content):
                        print(f"   ✓ 파싱 성공!")
                        return True
                else:
                    print(f"   HTML 응답 (RSS 아님)")
            else:
                print(f"   실패: {response.status_code}")
                
        except requests.exceptions.RequestException as e:
            print(f"   요청 오류: {e}")
        except Exception as e:
            print(f"   예외: {e}")
    
    return False

def selenium_complete_example():
    """완전한 Selenium 예제 코드 생성"""
    
    selenium_code = '''
# 완전한 Selenium 접근 방법
# pip install selenium 필요

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.action_chains import ActionChains
import json
import time
import pandas as pd

def get_g2b_data_selenium():
    """Selenium으로 나라장터 데이터 수집"""
    
    # Chrome 옵션 설정
    options = webdriver.ChromeOptions()
    # options.add_argument('--headless')  # 백그라운드 실행 (디버깅시 주석처리)
    options.add_argument('--no-sandbox')
    options.add_argument('--disable-dev-shm-usage')
    options.add_argument('--disable-blink-features=AutomationControlled')
    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    options.add_experimental_option('useAutomationExtension', False)
    
    driver = webdriver.Chrome(options=options)
    driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")
    
    try:
        print("1. 나라장터 입찰공고 페이지 접속...")
        driver.get('https://www.g2b.go.kr/ep/tbid/tbidList.do')
        
        # 페이지 로딩 완료 대기
        WebDriverWait(driver, 15).until(
            EC.presence_of_element_located((By.CLASS_NAME, "search_form"))
        )
        
        print("2. 검색 조건 설정...")
        
        # 검색어 입력 (예: 인공지능)
        try:
            search_input = WebDriverWait(driver, 10).until(
                EC.element_to_be_clickable((By.NAME, "bidPbancNm"))
            )
            search_input.clear()
            search_input.send_keys("인공지능")
            print("   검색어 입력 완료")
        except:
            print("   검색어 입력란을 찾을 수 없습니다.")
        
        # 날짜 설정
        try:
            from_date = driver.find_element(By.NAME, "fromBidDt")
            to_date = driver.find_element(By.NAME, "toBidDt")
            
            from_date.clear()
            from_date.send_keys("2024-12-01")
            
            to_date.clear()
            to_date.send_keys("2024-12-31")
            print("   날짜 설정 완료")
        except:
            print("   날짜 입력란을 찾을 수 없습니다.")
        
        # 검색 버튼 클릭
        try:
            search_btn = WebDriverWait(driver, 10).until(
                EC.element_to_be_clickable((By.CLASS_NAME, "btn_search"))
            )
            search_btn.click()
            print("3. 검색 실행...")
            
            # 결과 로딩 대기
            WebDriverWait(driver, 15).until(
                EC.presence_of_element_located((By.CLASS_NAME, "search_result"))
            )
            
        except Exception as e:
            print(f"   검색 버튼 클릭 실패: {e}")
            return False
        
        print("4. 결과 데이터 추출...")
        
        # 결과 테이블에서 데이터 추출
        results = []
        try:
            # 결과 행들 찾기
            rows = driver.find_elements(By.CSS_SELECTOR, "table.search_result tbody tr")
            
            for i, row in enumerate(rows):
                try:
                    cells = row.find_elements(By.TAG_NAME, "td")
                    if len(cells) >= 5:  # 충분한 열이 있는 경우만
                        result_data = {
                            'index': i + 1,
                            'bid_name': cells[1].text.strip(),
                            'organization': cells[2].text.strip(),
                            'announcement_date': cells[3].text.strip(),
                            'bid_method': cells[4].text.strip(),
                            'status': cells[5].text.strip() if len(cells) > 5 else ''
                        }
                        results.append(result_data)
                        
                        # 첫 번째 결과 미리보기
                        if i == 0:
                            print("   첫 번째 결과:")
                            for key, value in result_data.items():
                                print(f"     {key}: {value}")
                
                except Exception as e:
                    print(f"   행 {i+1} 처리 오류: {e}")
                    continue
            
            print(f"   총 {len(results)}개 결과 추출")
            
            # JSON 저장
            with open('g2b_selenium_results.json', 'w', encoding='utf-8') as f:
                json.dump(results, f, ensure_ascii=False, indent=2)
            
            # Excel 저장
            if results:
                df = pd.DataFrame(results)
                df.to_excel('g2b_selenium_results.xlsx', index=False)
                print("   결과를 Excel 파일로도 저장했습니다.")
            
            return len(results) > 0
            
        except Exception as e:
            print(f"   데이터 추출 오류: {e}")
            return False
        
    except Exception as e:
        print(f"Selenium 실행 오류: {e}")
        return False
        
    finally:
        print("5. 브라우저 종료...")
        driver.quit()

# 실행
if __name__ == "__main__":
    success = get_g2b_data_selenium()
    if success:
        print("\\n✓ Selenium을 통한 데이터 수집 성공!")
    else:
        print("\\n✗ 데이터 수집 실패")
'''
    
    return selenium_code

# 실행
def main():
    print("나라장터 데이터 수집 결과 분석\n")
    
    # 1. 기존 RSS 파일 확인
    rss_success = check_rss_file()
    
    if not rss_success:
        # 2. 실제 RSS 피드 시도
        rss_success = try_proper_rss_feeds()
    
    if not rss_success:
        print("\n=== 최종 권장 방법: Selenium ===")
        selenium_code = selenium_complete_example()
        
        with open('selenium_g2b_scraper.py', 'w', encoding='utf-8') as f:
            f.write(selenium_code)
        
        print("완전한 Selenium 스크래퍼 코드를 'selenium_g2b_scraper.py'에 저장했습니다.")
        print("\n사용 방법:")
        print("1. pip install selenium pandas openpyxl")
        print("2. Chrome 드라이버 설치")
        print("3. python selenium_g2b_scraper.py")
    else:
        print("\n✓ RSS 데이터 수집 성공!")

if __name__ == "__main__":
    main()

나라장터 데이터 수집 결과 분석

=== RSS 파일 내용 분석 ===
파일 크기: 1021 bytes
처음 500자:
<!DOCTYPE html>

<html lang="ko">

<head>

    <meta charset="UTF-8">

    <meta http-equiv="X-UA-Compatible" content="IE=edge">

    <meta name="viewport" content="width=device-width, initial-scale=1.0">

    <title>ìì¤í ì ê·¼ ìë´</title>

    <link rel="stylesheet" href="/static/info/css/common.css">



</head>

<body class="wrapper error_404">

    <div class="conArea">

        <h1>ëë¼ì¥í° êµ­ê°ì¢í©ì ìì¡°ë¬</h1>

        <div class="conText">

             <div>

       

이것은 HTML 페이지입니다. RSS XML이 아닙니다.

=== 실제 RSS 피드 시도 ===

1. https://www.g2b.go.kr/rss/rss.do 시도 중...
   상태 코드: 200
   응답 길이: 1019 bytes
   응답 시작: <!DOCTYPE html>
<html lang="ko">
<head>
    <meta charset="UTF-8">
    <meta http-equiv="X-UA-Co...
   HTML 응답 (RSS 아님)

2. https://www.g2b.go.kr/rss/rss.do?category=bid 시도 중...
   상태 코드: 200
   응답 길이: 1019 bytes
   응답 시작: <!DOCTYPE html>
<html lang="ko">
<head>
    <meta charset="UTF-8

In [19]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.action_chains import ActionChains
from selenium.common.exceptions import TimeoutException, ElementClickInterceptedException
import time
import json

def setup_driver():
    """Chrome 드라이버 설정"""
    options = webdriver.ChromeOptions()
    
    # 실제 브라우저처럼 보이게 설정
    options.add_argument('--disable-blink-features=AutomationControlled')
    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    options.add_experimental_option('useAutomationExtension', False)
    options.add_argument('--no-sandbox')
    options.add_argument('--disable-dev-shm-usage')
    
    # 창 크기 설정
    options.add_argument('--window-size=1920,1080')
    
    # 헤드리스 모드 (디버깅할 때는 주석처리)
    # options.add_argument('--headless')
    
    driver = webdriver.Chrome(options=options)
    
    # 자동화 탐지 우회
    driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")
    
    return driver

def navigate_to_bid_list():
    """나라장터 메인페이지에서 입찰공고목록으로 이동"""
    
    driver = setup_driver()
    
    try:
        print("1. 나라장터 메인페이지 접속...")
        driver.get('https://www.g2b.go.kr/')
        
        # 페이지 로딩 대기
        WebDriverWait(driver, 15).until(
            EC.presence_of_element_located((By.TAG_NAME, "body"))
        )
        
        print("2. 페이지 로딩 완료, 메뉴 네비게이션 시작...")
        
        # ActionChains로 마우스 움직임 제어
        actions = ActionChains(driver)
        
        # 1단계: "입찰" 메뉴에 마우스 오버
        print("3. '입찰' 메뉴에 마우스 오버...")
        
        # 여러 방법으로 입찰 메뉴 찾기 시도
        bid_menu_selectors = [
            "//span[contains(text(), '입찰')]",
            "//a[contains(@class, 'depth1')]//span[contains(text(), '입찰')]",
            "#mf_wfm_gnb_wfm_gnbMenu_genDepth1_1_btn_menuLvl1",
            "a[class*='depth1'] span:contains('입찰')"
        ]
        
        bid_menu = None
        for selector in bid_menu_selectors:
            try:
                if selector.startswith("//"):
                    bid_menu = WebDriverWait(driver, 5).until(
                        EC.presence_of_element_located((By.XPATH, selector))
                    )
                elif selector.startswith("#"):
                    bid_menu = WebDriverWait(driver, 5).until(
                        EC.presence_of_element_located((By.ID, selector.replace("#", "")))
                    )
                else:
                    bid_menu = WebDriverWait(driver, 5).until(
                        EC.presence_of_element_located((By.CSS_SELECTOR, selector))
                    )
                print(f"   '입찰' 메뉴 찾음: {selector}")
                break
            except TimeoutException:
                continue
        
        if not bid_menu:
            print("   '입찰' 메뉴를 찾을 수 없습니다.")
            return False
        
        # 입찰 메뉴에 마우스 오버
        actions.move_to_element(bid_menu).perform()

        # DOM 업데이트 기다리기 (입찰공고 메뉴 등장)
        try:
            bid_announcement = WebDriverWait(driver, 10).until(
                EC.visibility_of_element_located(
                    (By.XPATH, "//a//span[contains(text(), '입찰공고')]")
                )
            )
            print("   '입찰공고' 서브메뉴 찾음")
        except TimeoutException:
            print("   '입찰공고' 서브메뉴를 찾을 수 없습니다.")
            return False

        # 입찰공고 서브메뉴에 마우스 오버
        actions.move_to_element(bid_announcement).perform()
        time.sleep(1)
        
        print("5. '입찰공고목록' 클릭...")
        
        # 3단계: "입찰공고목록" 클릭
        bid_list_selectors = [
            "//span[contains(text(), '입찰공고목록')]",
            "#mf_wfm_gnb_wfm_gnbMenu_genDepth1_1_genDepth2_0_genDepth3_0_btn_menuLvl3",
            "//a[contains(@class, 'depth3')]//span[contains(text(), '입찰공고목록')]"
        ]
        
        bid_list = None
        for selector in bid_list_selectors:
            try:
                if selector.startswith("//"):
                    bid_list = WebDriverWait(driver, 5).until(
                        EC.element_to_be_clickable((By.XPATH, selector))
                    )
                else:
                    bid_list = WebDriverWait(driver, 5).until(
                        EC.element_to_be_clickable((By.ID, selector.replace("#", "")))
                    )
                print(f"   '입찰공고목록' 찾음: {selector}")
                break
            except TimeoutException:
                continue
        
        if not bid_list:
            print("   '입찰공고목록'을 찾을 수 없습니다.")
            return False
        
        # 입찰공고목록 클릭
        try:
            # JavaScript 클릭 시도
            driver.execute_script("arguments[0].click();", bid_list)
            print("   JavaScript로 클릭 성공")
        except Exception:
            try:
                # 일반 클릭 시도
                bid_list.click()
                print("   일반 클릭 성공")
            except ElementClickInterceptedException:
                # ActionChains 클릭 시도
                actions.click(bid_list).perform()
                print("   ActionChains 클릭 성공")
        
        # 페이지 이동 대기
        print("6. 입찰공고목록 페이지 로딩 대기...")
        WebDriverWait(driver, 15).until(
            lambda d: "tbidList" in d.current_url or "입찰" in d.title
        )
        
        print(f"   현재 URL: {driver.current_url}")
        print(f"   페이지 제목: {driver.title}")
        
        return driver  # 드라이버를 반환해서 계속 사용할 수 있도록
        
    except Exception as e:
        print(f"메뉴 네비게이션 오류: {e}")
        driver.quit()
        return False

def search_and_extract_data(driver, search_term="", from_date="", to_date=""):
    """검색 실행 및 데이터 추출"""
    
    try:
        print("7. 검색 조건 설정...")
        
        # 검색어 입력
        if search_term:
            try:
                search_input = WebDriverWait(driver, 10).until(
                    EC.presence_of_element_located((By.NAME, "bidPbancNm"))
                )
                search_input.clear()
                search_input.send_keys(search_term)
                print(f"   검색어 '{search_term}' 입력 완료")
            except TimeoutException:
                print("   검색어 입력란을 찾을 수 없습니다.")
        
        # 날짜 설정
        if from_date and to_date:
            try:
                # 공고일 라디오 버튼 선택
                date_radio = driver.find_element(By.XPATH, "//input[@name='bidDateType' and @value='R']")
                driver.execute_script("arguments[0].click();", date_radio)
                
                # 시작 날짜
                from_date_input = driver.find_element(By.NAME, "fromBidDt")
                from_date_input.clear()
                from_date_input.send_keys(from_date)
                
                # 종료 날짜
                to_date_input = driver.find_element(By.NAME, "toBidDt")
                to_date_input.clear()
                to_date_input.send_keys(to_date)
                
                print(f"   날짜 설정 완료: {from_date} ~ {to_date}")
            except Exception as e:
                print(f"   날짜 설정 오류: {e}")
        
        # 검색 버튼 클릭
        search_btn_selectors = [
            "//input[@value='검색']",
            "//button[contains(text(), '검색')]",
            ".btn_search",
            "#searchBtn"
        ]
        
        search_btn = None
        for selector in search_btn_selectors:
            try:
                if selector.startswith("//"):
                    search_btn = driver.find_element(By.XPATH, selector)
                else:
                    search_btn = driver.find_element(By.CSS_SELECTOR, selector)
                break
            except:
                continue
        
        if search_btn:
            driver.execute_script("arguments[0].click();", search_btn)
            print("8. 검색 실행...")
            
            # 결과 로딩 대기
            time.sleep(3)
            
            # 결과 테이블 확인
            try:
                result_table = WebDriverWait(driver, 10).until(
                    EC.presence_of_element_located((By.TAG_NAME, "table"))
                )
                print("   검색 결과 테이블 발견")
                
                # 데이터 추출 로직은 다음 단계에서...
                return True
                
            except TimeoutException:
                print("   검색 결과를 찾을 수 없습니다.")
                return False
        else:
            print("   검색 버튼을 찾을 수 없습니다.")
            return False
    
    except Exception as e:
        print(f"검색 오류: {e}")
        return False

def main():
    """메인 실행 함수"""
    print("=== 나라장터 Selenium 네비게이션 테스트 ===\n")
    
    # 메뉴 네비게이션
    driver = navigate_to_bid_list()
    
    if driver:
        print("\n✓ 메뉴 네비게이션 성공!")
        
        # 검색 실행 (예시)
        search_success = search_and_extract_data(
            driver, 
            search_term="인공지능",
            from_date="2024-12-01",
            to_date="2024-12-31"
        )
        
        if search_success:
            print("\n✓ 검색 실행 성공!")
        else:
            print("\n✗ 검색 실행 실패")
        
        # 5초 대기 후 종료 (결과 확인용)
        print("\n5초 후 브라우저를 종료합니다...")
        time.sleep(5)
        driver.quit()
        
    else:
        print("\n✗ 메뉴 네비게이션 실패")

if __name__ == "__main__":
    main()

=== 나라장터 Selenium 네비게이션 테스트 ===

1. 나라장터 메인페이지 접속...
2. 페이지 로딩 완료, 메뉴 네비게이션 시작...
3. '입찰' 메뉴에 마우스 오버...
   '입찰' 메뉴 찾음: //span[contains(text(), '입찰')]


KeyboardInterrupt: 

In [17]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.action_chains import ActionChains
from selenium.common.exceptions import TimeoutException, ElementClickInterceptedException
import time


def setup_driver():
    """Chrome 드라이버 설정"""
    options = webdriver.ChromeOptions()
    options.add_argument('--disable-blink-features=AutomationControlled')
    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    options.add_experimental_option('useAutomationExtension', False)
    options.add_argument('--no-sandbox')
    options.add_argument('--disable-dev-shm-usage')
    options.add_argument('--window-size=1920,1080')
    # options.add_argument('--headless')  # 디버깅할 때는 주석처리
    driver = webdriver.Chrome(options=options)
    driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")
    return driver


def close_layer_popup(driver):
    """나라장터 메인 진입 후 뜨는 레이어 팝업 닫기"""
    try:
        close_btn = WebDriverWait(driver, 3).until(
            EC.element_to_be_clickable(
                (By.CSS_SELECTOR, "div.popup a.close, .layerPopup .btn_close, .popupWrap .close")
            )
        )
        close_btn.click()
        print("   레이어 팝업 닫음")
        time.sleep(0.5)
    except TimeoutException:
        print("   닫을 레이어 팝업 없음")


def navigate_to_bid_list():
    """나라장터 메인페이지에서 입찰공고목록으로 이동"""
    driver = setup_driver()

    try:
        print("1. 나라장터 메인페이지 접속...")
        driver.get('https://www.g2b.go.kr/')

        WebDriverWait(driver, 15).until(
            EC.presence_of_element_located((By.TAG_NAME, "body"))
        )
        print("2. 페이지 로딩 완료")

        # 팝업 닫기 시도
        close_layer_popup(driver)

        print("3. 메뉴 네비게이션 시작...")
        actions = ActionChains(driver)

        # 1단계: '입찰' 메뉴
        bid_menu = WebDriverWait(driver, 10).until(
            EC.visibility_of_element_located((By.XPATH, "//span[contains(text(), '입찰')]"))
        )
        actions.move_to_element(bid_menu).perform()
        print("   '입찰' 메뉴에 마우스 오버")

        # 2단계: '입찰공고'
        bid_announcement = WebDriverWait(driver, 10).until(
            EC.visibility_of_element_located((By.XPATH, "//a//span[contains(text(), '입찰공고')]"))
        )
        actions.move_to_element(bid_announcement).perform()
        print("   '입찰공고' 서브메뉴에 마우스 오버")

        time.sleep(0.5)

        # 3단계: '입찰공고목록'
        bid_list = WebDriverWait(driver, 10).until(
            EC.element_to_be_clickable((By.XPATH, "//a//span[contains(text(), '입찰공고목록')]"))
        )
        driver.execute_script("arguments[0].click();", bid_list)
        print("   '입찰공고목록' 클릭 완료")

        # 페이지 이동 대기
        WebDriverWait(driver, 15).until(
            lambda d: "bid" in d.current_url or "입찰" in d.title
        )

        print(f"   현재 URL: {driver.current_url}")
        print(f"   페이지 제목: {driver.title}")

        return driver

    except Exception as e:
        print(f"메뉴 네비게이션 오류: {e}")
        driver.quit()
        return False


def search_and_extract_data(driver, search_term="", from_date="", to_date=""):
    """검색 실행 및 데이터 추출"""
    try:
        print("4. 검색 조건 설정...")

        # 검색어 입력
        if search_term:
            try:
                search_input = WebDriverWait(driver, 10).until(
                    EC.presence_of_element_located((By.NAME, "bidPbancNm"))
                )
                search_input.clear()
                search_input.send_keys(search_term)
                print(f"   검색어 '{search_term}' 입력 완료")
            except TimeoutException:
                print("   검색어 입력란을 찾을 수 없습니다.")

        # 날짜 입력
        if from_date and to_date:
            try:
                date_radio = driver.find_element(By.XPATH, "//input[@name='bidDateType' and @value='R']")
                driver.execute_script("arguments[0].click();", date_radio)

                from_date_input = driver.find_element(By.NAME, "fromBidDt")
                from_date_input.clear()
                from_date_input.send_keys(from_date)

                to_date_input = driver.find_element(By.NAME, "toBidDt")
                to_date_input.clear()
                to_date_input.send_keys(to_date)

                print(f"   날짜 설정 완료: {from_date} ~ {to_date}")
            except Exception as e:
                print(f"   날짜 설정 오류: {e}")

        # 검색 버튼 클릭
        search_btn = None
        try:
            search_btn = driver.find_element(By.XPATH, "//input[@value='검색']")
        except:
            try:
                search_btn = driver.find_element(By.CSS_SELECTOR, ".btn_search")
            except:
                pass

        if search_btn:
            driver.execute_script("arguments[0].click();", search_btn)
            print("5. 검색 실행...")

            time.sleep(3)

            try:
                result_table = WebDriverWait(driver, 10).until(
                    EC.presence_of_element_located((By.TAG_NAME, "table"))
                )
                print("   검색 결과 테이블 발견 ✅")
                return True
            except TimeoutException:
                print("   검색 결과를 찾을 수 없습니다.")
                return False
        else:
            print("   검색 버튼을 찾을 수 없습니다.")
            return False

    except Exception as e:
        print(f"검색 오류: {e}")
        return False


def main():
    """메인 실행 함수"""
    print("=== 나라장터 Selenium 자동화 실행 ===\n")

    driver = navigate_to_bid_list()

    if driver:
        print("\n✓ 메뉴 네비게이션 성공!")

        search_success = search_and_extract_data(
            driver,
            search_term="인공지능",
            from_date="2024-12-01",
            to_date="2024-12-31"
        )

        if search_success:
            print("\n✓ 검색 실행 성공!")
        else:
            print("\n✗ 검색 실행 실패")

        print("\n5초 후 브라우저 종료...")
        time.sleep(5)
        driver.quit()
    else:
        print("\n✗ 메뉴 네비게이션 실패")


if __name__ == "__main__":
    main()


=== 나라장터 Selenium 자동화 실행 ===

1. 나라장터 메인페이지 접속...
2. 페이지 로딩 완료
   닫을 레이어 팝업 없음
3. 메뉴 네비게이션 시작...
   '입찰' 메뉴에 마우스 오버
메뉴 네비게이션 오류: Message: 


✗ 메뉴 네비게이션 실패


In [5]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException, ElementClickInterceptedException
import time

def close_popups(driver):
    """w2ui 팝업 강제 종료 (refid 기준)"""
    try:
        buttons = driver.find_elements(By.CSS_SELECTOR, "button.w2window_close.w2window_close_acc")
        print(f"발견된 팝업 닫기 버튼 개수: {len(buttons)}")

        for i, btn in enumerate(buttons, start=1):
            refid = btn.get_attribute("refid")
            if refid:
                # w2ui 내부 함수 호출로 팝업 종료
                driver.execute_script(f"window.$w2 && $w2.closeWindow('{refid}');")
                print(f"{i}번째 팝업 (refid={refid}) 닫기 성공")
            else:
                # refid 없으면 fallback
                driver.execute_script("arguments[0].click();", btn)
                print(f"{i}번째 팝업 닫기 fallback click")

            time.sleep(0.5)

    except Exception as e:
        print(f"팝업 닫기 실패: {e}")

def navigate_to_bid_list(driver):
    """나라장터 메인페이지에서 입찰 → 입찰공고 → 입찰공고목록 진입"""
    wait = WebDriverWait(driver, 10)
    actions = ActionChains(driver)

    # 팝업 먼저 닫기
    close_popups(driver)

    try:
        # '입찰' 메뉴 찾기
        bid_menu = wait.until(EC.presence_of_element_located((By.XPATH, "//span[contains(text(), '입찰')]")))
        print("'입찰' 메뉴 찾음")
        actions.move_to_element(bid_menu).perform()
        time.sleep(1)

        # '입찰공고' 서브메뉴 찾기
        bid_notice = wait.until(EC.presence_of_element_located((By.XPATH, "//a[contains(text(), '입찰공고')]")))
        print("'입찰공고' 서브메뉴 찾음")
        actions.move_to_element(bid_notice).perform()
        time.sleep(1)

        # '입찰공고목록' 클릭
        bid_list = wait.until(EC.presence_of_element_located((By.XPATH, "//a[contains(text(), '입찰공고목록')]")))
        driver.execute_script("arguments[0].click();", bid_list)
        print("'입찰공고목록' 클릭 완료")

    except Exception as e:
        print(f"메뉴 네비게이션 실패: {e}")

if __name__ == "__main__":
    driver = webdriver.Chrome()
    driver.get("https://www.g2b.go.kr/")
    driver.maximize_window()

    time.sleep(5)

    print("=== 나라장터 Selenium 네비게이션 테스트 ===")
    navigate_to_bid_list(driver)


=== 나라장터 Selenium 네비게이션 테스트 ===
발견된 팝업 닫기 버튼 개수: 5
1번째 팝업 (refid=mf_wfm_container_wq_uuid_915_wq_uuid_922_poupR23AB00000134241) 닫기 성공
2번째 팝업 (refid=mf_wfm_container_wq_uuid_915_wq_uuid_922_poupR23AB00000134324) 닫기 성공
3번째 팝업 (refid=mf_wfm_container_wq_uuid_915_wq_uuid_922_poupR23AB00000134343) 닫기 성공
4번째 팝업 (refid=mf_wfm_container_wq_uuid_915_wq_uuid_922_poupR23AB00000134350) 닫기 성공
5번째 팝업 (refid=mf_wfm_container_wq_uuid_915_wq_uuid_922_poupR23AB00000134353) 닫기 성공
'입찰' 메뉴 찾음
메뉴 네비게이션 실패: Message: 
Stacktrace:
	GetHandleVerifier [0x0x7ff70dbc30f5+79493]
	GetHandleVerifier [0x0x7ff70dbc3150+79584]
	(No symbol) [0x0x7ff70d9401ba]
	(No symbol) [0x0x7ff70d998067]
	(No symbol) [0x0x7ff70d99832c]
	(No symbol) [0x0x7ff70d9ebe27]
	(No symbol) [0x0x7ff70d9c074f]
	(No symbol) [0x0x7ff70d9e8b8b]
	(No symbol) [0x0x7ff70d9c04e3]
	(No symbol) [0x0x7ff70d988e92]
	(No symbol) [0x0x7ff70d989c63]
	GetHandleVerifier [0x0x7ff70de80dbd+2954061]
	GetHandleVerifier [0x0x7ff70de7b02a+2930106]
	GetHandleVerifier [

In [27]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.action_chains import ActionChains
import time
import json
import pandas as pd

def setup_driver():
    """Chrome 드라이버 설정"""
    options = webdriver.ChromeOptions()
    options.add_argument('--disable-blink-features=AutomationControlled')
    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    options.add_experimental_option('useAutomationExtension', False)
    options.add_argument('--no-sandbox')
    options.add_argument('--disable-dev-shm-usage')
    
    driver = webdriver.Chrome(options=options)
    driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")
    
    return driver

def navigate_to_bid_list():
    """나라장터 입찰공고목록 페이지로 이동"""
    
    driver = setup_driver()
    
    try:
        print("1. 나라장터 메인페이지 접속...")
        driver.get("https://www.g2b.go.kr/")
        driver.maximize_window()
        wait = WebDriverWait(driver, 15)

        # 페이지 로딩 대기
        time.sleep(5)

        
        print("3. '입찰' 메뉴에 마우스 오버...")
        
        # 정확한 XPath를 우선으로 시도
        bid_menu_selectors = [
            "//*[@id='mf_wfm_gnb_wfm_gnbMenu_genDepth1_1_btn_menuLvl1_span']",
            "//span[contains(text(), '입찰')]//parent::a[@class*='depth1']",
            "//a[contains(@class, 'depth1')]//span[text()='입찰']",
            "//span[text()='입찰']"
        ]
        
        element_to_hover = None
        actions = ActionChains(driver)
        
        for selector in bid_menu_selectors:
            try:
                if selector.startswith("//") or selector.startswith("//*"):
                    element_to_hover = driver.find_element(By.XPATH, selector)
                    print(f"   '입찰' 메뉴 발견 (XPath): {selector}")
                else:
                    element_to_hover = driver.find_element(By.CSS_SELECTOR, selector)
                    print(f"   '입찰' 메뉴 발견 (CSS): {selector}")
                
                if element_to_hover:
                    break
            except Exception as e:
                print(f"   선택자 {selector} 실패: {e}")
                continue
        
        if not element_to_hover:
            print("   '입찰' 메뉴를 찾을 수 없습니다. 다른 방법 시도...")
            # 직접 URL로 이동
            driver.get("https://www.g2b.go.kr/ep/tbid/tbidList.do")
            time.sleep(3)
            print("   직접 입찰공고 페이지로 이동")
            return driver
        
        # 마우스 오버 실행
        actions.move_to_element(element_to_hover).perform()
        
        # 잠시 대기
        time.sleep(2)
        
        print("4. '입찰공고목록' 메뉴 클릭...")

        try:
            # 정확한 XPath를 우선으로 시도
            bid_list_selectors = [
                "//*[@id='mf_wfm_gnb_wfm_gnbMenu_genDepth1_1_genDepth2_0_genDepth3_0_btn_menuLvl3_span']",
                "//span[contains(text(), '입찰공고목록')]",
                "//a[contains(@class, 'depth3')]//span[text()='입찰공고목록']",
                "#mf_wfm_gnb_wfm_gnbMenu_genDepth1_1_genDepth2_0_genDepth3_0_btn_menuLvl3_span"
            ]
            
            bid_list_element = None
            
            for selector in bid_list_selectors:
                try:
                    if selector.startswith("//") or selector.startswith("//*"):
                        bid_list_element = wait.until(
                            EC.element_to_be_clickable((By.XPATH, selector))
                        )
                        print(f"   '입찰공고목록' 메뉴 발견 (XPath): {selector}")
                    elif selector.startswith("#"):
                        bid_list_element = wait.until(
                            EC.element_to_be_clickable((By.ID, selector.replace("#", "")))
                        )
                        print(f"   '입찰공고목록' 메뉴 발견 (ID): {selector}")
                    
                    if bid_list_element:
                        break
                        
                except Exception as e:
                    print(f"   선택자 {selector} 실패: {e}")
                    continue
            
            if bid_list_element:
                # JS로 강제 클릭
                driver.execute_script("arguments[0].click();", bid_list_element)
                print("✓ '입찰공고목록' 메뉴 클릭 완료")
            else:
                print("   '입찰공고목록' 메뉴를 찾을 수 없습니다. 직접 URL로 이동...")
                driver.get("https://www.g2b.go.kr/ep/tbid/tbidList.do")

            # 페이지 이동 잠시 대기
            time.sleep(3)
            print(f"현재 URL: {driver.current_url}")
            print(f"페이지 제목: {driver.title}")
            
            return driver

        except Exception as e:
            print(f"메뉴 진입 실패: {e}")
            print("직접 URL로 이동 시도...")
            driver.get("https://www.g2b.go.kr/ep/tbid/tbidList.do")
            time.sleep(3)
            return driver
            
    except Exception as e:
        print(f"페이지 접속 실패: {e}")
        driver.quit()
        return None

def search_and_extract_data(driver, search_term="인공지능", from_date="2024-12-01", to_date="2024-12-31"):
    """검색 조건 설정 및 데이터 추출"""
    
    try:
        print(f"5. 검색 조건 설정 (검색어: '{search_term}', 기간: {from_date}~{to_date})...")
        wait = WebDriverWait(driver, 10)
        
        # 검색어 입력
        try:
            search_input = wait.until(EC.presence_of_element_located((By.NAME, "bidPbancNm")))
            search_input.clear()
            search_input.send_keys(search_term)
            print(f"   검색어 '{search_term}' 입력 완료")
        except Exception as e:
            print(f"   검색어 입력 실패: {e}")
        
        # 공고일 기준으로 설정 (R = 공고일)
        try:
            date_radio = driver.find_element(By.XPATH, "//input[@name='bidDateType' and @value='R']")
            driver.execute_script("arguments[0].click();", date_radio)
            print("   공고일 기준으로 설정")
        except Exception as e:
            print(f"   날짜 타입 설정 실패: {e}")
        
        # 시작 날짜 입력
        try:
            from_date_input = driver.find_element(By.NAME, "fromBidDt")
            from_date_input.clear()
            from_date_input.send_keys(from_date.replace("-", ""))  # YYYYMMDD 형식
            print(f"   시작 날짜: {from_date}")
        except Exception as e:
            print(f"   시작 날짜 입력 실패: {e}")
        
        # 종료 날짜 입력
        try:
            to_date_input = driver.find_element(By.NAME, "toBidDt")
            to_date_input.clear()
            to_date_input.send_keys(to_date.replace("-", ""))  # YYYYMMDD 형식
            print(f"   종료 날짜: {to_date}")
        except Exception as e:
            print(f"   종료 날짜 입력 실패: {e}")
        
        # 검색 버튼 클릭
        print("6. 검색 실행...")
        search_buttons = [
            "//input[@type='button' and @value='검색']",
            "//button[contains(text(), '검색')]",
            "//input[contains(@class, 'btn') and @value='검색']",
            ".btn_search"
        ]
        
        search_clicked = False
        for selector in search_buttons:
            try:
                if selector.startswith("//"):
                    search_btn = driver.find_element(By.XPATH, selector)
                else:
                    search_btn = driver.find_element(By.CSS_SELECTOR, selector)
                
                driver.execute_script("arguments[0].click();", search_btn)
                print("   검색 버튼 클릭 완료")
                search_clicked = True
                break
            except Exception:
                continue
        
        if not search_clicked:
            print("   검색 버튼을 찾을 수 없습니다.")
            return []
        
        # 결과 로딩 대기
        time.sleep(5)
        
        print("7. 검색 결과 추출...")
        
        # 결과 데이터 추출
        results = []
        
        try:
            # 결과 테이블 찾기 (여러 선택자 시도)
            table_selectors = [
                "table.search_result",
                "table[summary*='입찰']",
                "table.list_table",
                "div.list_area table",
                "tbody tr"
            ]
            
            rows = []
            for selector in table_selectors:
                try:
                    if selector == "tbody tr":
                        rows = driver.find_elements(By.CSS_SELECTOR, selector)
                    else:
                        table = driver.find_element(By.CSS_SELECTOR, selector)
                        rows = table.find_elements(By.CSS_SELECTOR, "tbody tr")
                    
                    if rows and len(rows) > 1:  # 헤더 제외하고 데이터가 있으면
                        print(f"   테이블 발견: {selector}")
                        break
                except Exception:
                    continue
            
            if not rows:
                print("   결과 테이블을 찾을 수 없습니다.")
                return []
            
            print(f"   총 {len(rows)}개 행 발견")
            
            # 각 행에서 데이터 추출
            for i, row in enumerate(rows[:20]):  # 최대 20개만
                try:
                    cells = row.find_elements(By.TAG_NAME, "td")
                    
                    if len(cells) >= 4:  # 최소 4개 열이 있어야 함
                        # 일반적인 나라장터 테이블 구조에 맞춰 추출
                        result_data = {
                            'index': i + 1,
                            'bid_number': cells[0].text.strip() if len(cells) > 0 else '',
                            'bid_name': cells[1].text.strip() if len(cells) > 1 else '',
                            'organization': cells[2].text.strip() if len(cells) > 2 else '',
                            'announcement_date': cells[3].text.strip() if len(cells) > 3 else '',
                            'bid_method': cells[4].text.strip() if len(cells) > 4 else '',
                            'status': cells[5].text.strip() if len(cells) > 5 else '',
                            'estimated_price': cells[6].text.strip() if len(cells) > 6 else ''
                        }
                        
                        # 빈 값이 아닌 경우만 추가
                        if any(value.strip() for value in result_data.values() if isinstance(value, str)):
                            results.append(result_data)
                            
                            # 첫 번째 결과 미리보기
                            if i == 0:
                                print("   첫 번째 검색 결과:")
                                for key, value in result_data.items():
                                    if value:
                                        print(f"     {key}: {value}")
                
                except Exception as e:
                    print(f"   행 {i+1} 처리 중 오류: {e}")
                    continue
            
            print(f"   총 {len(results)}개 유효한 결과 추출")
            
            if results:
                # JSON 파일로 저장
                with open('g2b_selenium_results.json', 'w', encoding='utf-8') as f:
                    json.dump(results, f, ensure_ascii=False, indent=2)
                print("   결과를 'g2b_selenium_results.json'에 저장")
                
                # Excel 파일로 저장
                try:
                    df = pd.DataFrame(results)
                    df.to_excel('g2b_selenium_results.xlsx', index=False)
                    print("   결과를 'g2b_selenium_results.xlsx'에 저장")
                except Exception as e:
                    print(f"   Excel 저장 실패: {e}")
            
            return results
            
        except Exception as e:
            print(f"   데이터 추출 오류: {e}")
            return []
    
    except Exception as e:
        print(f"검색 및 데이터 추출 실패: {e}")
        return []

def main():
    """메인 실행 함수"""
    print("=== 나라장터 Selenium 데이터 수집 시작 ===\n")
    
    # 1. 메뉴 네비게이션
    driver = navigate_to_bid_list()
    
    if not driver:
        print("메뉴 진입에 실패했습니다.")
        return
    
    try:
        # 2. 검색 및 데이터 추출
        results = search_and_extract_data(driver)
        
        if results:
            print(f"\n✓ 성공적으로 {len(results)}개의 입찰 정보를 수집했습니다!")
        else:
            print("\n검색 결과가 없거나 데이터 추출에 실패했습니다.")
        
        # 결과 확인을 위해 잠시 대기
        print("\n5초 후 브라우저를 종료합니다...")
        time.sleep(5)
        
    finally:
        driver.quit()
        print("브라우저 종료 완료")

if __name__ == "__main__":
    main()

=== 나라장터 Selenium 데이터 수집 시작 ===

1. 나라장터 메인페이지 접속...
3. '입찰' 메뉴에 마우스 오버...
   '입찰' 메뉴 발견 (XPath): //*[@id='mf_wfm_gnb_wfm_gnbMenu_genDepth1_1_btn_menuLvl1_span']
4. '입찰공고목록' 메뉴 클릭...
   선택자 //*[@id='mf_wfm_gnb_wfm_gnbMenu_genDepth1_1_genDepth2_0_genDepth3_0_btn_menuLvl3_span'] 실패: Message: 

   선택자 //span[contains(text(), '입찰공고목록')] 실패: Message: 

   선택자 //a[contains(@class, 'depth3')]//span[text()='입찰공고목록'] 실패: Message: 

   선택자 #mf_wfm_gnb_wfm_gnbMenu_genDepth1_1_genDepth2_0_genDepth3_0_btn_menuLvl3_span 실패: Message: 

   '입찰공고목록' 메뉴를 찾을 수 없습니다. 직접 URL로 이동...
현재 URL: https://www.g2b.go.kr/ep/tbid/tbidList.do
페이지 제목: 시스템 접근 안내
5. 검색 조건 설정 (검색어: '인공지능', 기간: 2024-12-01~2024-12-31)...
브라우저 종료 완료


KeyboardInterrupt: 

In [29]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
import time
import json

# Chrome 설정
chrome_options = Options()
chrome_options.add_argument("--start-maximized")
chrome_options.add_experimental_option("excludeSwitches", ["enable-logging"])

driver = webdriver.Chrome(options=chrome_options)

# 1. 나라장터 입찰공고목록 URL 접속
url = "https://www.g2b.go.kr/"
driver.get(url)

# 페이지 로딩 대기
time.sleep(3)  # 필요시 WebDriverWait로 변경 가능

# 2. 로그인 후 화면에서 JS 실행
# 기본 화면 로딩
driver.execute_script("scwin.onpageload();")

# 3. 검색 조건 paramData 설정
param_data = {
    "fromBidDt": "2025-09-01",
    "toBidDt": "2025-09-23",
    "callbackFn": "scwin.fnSrch4",
    "bidPbancNo": "",
    "odnLmtLgdngCd": "00"  # 전국
}

# paramData 브라우저 JS에 세팅
driver.execute_script(f'$p.setParameter("param", {json.dumps(param_data)});')

# 4. 화면 로딩 후 목록 조회
driver.execute_script("scwin.onpageload();")  # reload with paramData
driver.execute_script("scwin.fnSrch4(1);")    # 1페이지 조회

# 5. 잠시 대기 후 결과 확인
time.sleep(5)

# 예: 결과 테이블 HTML 가져오기
table_html = driver.execute_script("return document.querySelector('table').outerHTML;")
print(table_html)

# 종료
# driver.quit()


JavascriptException: Message: javascript error: scwin.onpageload is not a function
  (Session info: chrome=140.0.7339.128)
Stacktrace:
	GetHandleVerifier [0x0x7ff70dbc30f5+79493]
	GetHandleVerifier [0x0x7ff70dbc3150+79584]
	(No symbol) [0x0x7ff70d9401ba]
	(No symbol) [0x0x7ff70d947cfe]
	(No symbol) [0x0x7ff70d94b131]
	(No symbol) [0x0x7ff70d9e9f1b]
	(No symbol) [0x0x7ff70d9c070a]
	(No symbol) [0x0x7ff70d9e8b8b]
	(No symbol) [0x0x7ff70d9c04e3]
	(No symbol) [0x0x7ff70d988e92]
	(No symbol) [0x0x7ff70d989c63]
	GetHandleVerifier [0x0x7ff70de80dbd+2954061]
	GetHandleVerifier [0x0x7ff70de7b02a+2930106]
	GetHandleVerifier [0x0x7ff70de9b357+3061991]
	GetHandleVerifier [0x0x7ff70dbdd60e+187294]
	GetHandleVerifier [0x0x7ff70dbe557f+219919]
	GetHandleVerifier [0x0x7ff70dbcc294+116772]
	GetHandleVerifier [0x0x7ff70dbcc449+117209]
	GetHandleVerifier [0x0x7ff70dbb2618+11176]
	BaseThreadInitThunk [0x0x7fff77c7259d+29]
	RtlUserThreadStart [0x0x7fff7996af78+40]


In [42]:
import requests
import json

# ==========================
# 1️⃣ 세션 생성
# ==========================
session = requests.Session()
base_url = "https://www.g2b.go.kr"

# 세션 발급 요청 (getSession.do)
session_headers = {
    "Accept": "application/json",
    "Content-Type": "application/json;charset=UTF-8",
    "Referer": "https://www.g2b.go.kr/",
    "Origin": "https://www.g2b.go.kr",
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/140.0.0.0 Safari/537.36"
}

print("⏳ 세션 발급 중...")
session_response = session.post(f"{base_url}/co/coz/coza/util/getSession.do", headers=session_headers)
if session_response.status_code == 200:
    print("✅ 세션 발급 성공")
else:
    print("❌ 세션 발급 실패:", session_response.status_code)
    exit(1)

# ==========================
# 2️⃣ Excel 다운로드 요청
# ==========================
excel_url = f"{base_url}/pn/pnp/pnpe/BidPbac/selectBidPbacListExcel.do"

payload = {
    "dlBidPbancLstM": {
        "untyBidPbancNo": "",
        "bidPbancNo": "",
        "bidPbancOrd": "",
        "prcmBsneUntyNoOrd": "",
        "prcmBsneSeCd": "0000 조070001 조070002 조070003 조070004 조070005 민079999",
        "bidPbancNm": "",
        "pbancPstgDt": "",
        "ldocNoVal": "",
        "bidPrspPrce": "",
        "ctrtDmndRcptNo": "",
        "dmstcOvrsSeCd": "",
        "pbancKndCd": "공440002",
        "ctrtTyCd": "",
        "bidCtrtMthdCd": "",
        "scsbdMthdCd": "",
        "fromBidDt": "20250514",
        "toBidDt": "20250515",
        "minBidPrspPrce": "",
        "maxBidPrspPrce": "",
        "bsneAllYn": "Y",
        "frcpYn": "Y",
        "rsrvYn": "Y",
        "laseYn": "Y",
        "untyGrpGb": "",
        "dmstNm": "",
        "pbancPicNm": "",
        "odnLmtLgdngCd": "",
        "odnLmtLgdngNm": "",
        "intpCd": "",
        "intpNm": "",
        "dtlsPrnmNo": "",
        "dtlsPrnmNm": "",
        "slprRcptDdlnYn": "",
        "lcrtTyCd": "",
        "isMas": "",
        "isElpdt": "",
        "oderInstUntyGrpNo": "",
        "instSearchRangeYn": "",
        "esdacYn": "",
        "infoSysCd": "정010029",
        "contxtSeCd": "콘010006",
        "bidDateType": "R",
        "brcoOrgnCd": "",
        "deptOrgnCd": "",
        "isShop": "",
        "srchTy": "0",
        "cangParmVal": "untySrch001",
        "currentPage": "",
        "recordCountPerPage": "10",
        "startIndex": 1,
        "endIndex": 10,
        "excelHeaderText": ""
    }
}

excel_headers = {
    "Accept": "*/*",
    "Content-Type": "application/json;charset=UTF-8",
    "Referer": "https://www.g2b.go.kr/",
    "Origin": "https://www.g2b.go.kr",
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/140.0.0.0 Safari/537.36"
}

print("⏳ Excel 다운로드 중...")
response = session.post(excel_url, headers=excel_headers, json=payload)

if response.status_code == 200:
    with open("bid_list.xlsx", "wb") as f:
        f.write(response.content)
    print("✅ Excel 파일 저장 완료: bid_list.xlsx")
else:
    print("❌ Excel 다운로드 실패:", response.status_code)
    print(response.text)


⏳ 세션 발급 중...
✅ 세션 발급 성공
⏳ Excel 다운로드 중...
✅ Excel 파일 저장 완료: bid_list.xlsx
